# 人工介入（Human-in-the-Loop）

在某些场景中，Agent 自动执行的结果需要人工确认后才能继续进行。LangChain 通过 `interrupt_before` / `interrupt_after` 实现人工介入。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## interrupt_before——在工具执行前暂停

当 Agent 触发工具调用时，先暂停等待人工确认。

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver

@tool
def book_course(course_name: str, user_name: str) -> str:
    """预订课程"""
    return f"已为 {user_name} 预订《{course_name}》"

checkpointer = MemorySaver()
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model, tools=[book_course],
    checkpointer=checkpointer,
    interrupt_before=["tools"],  # 工具执行前暂停
    system_prompt="你是课程预订助手。",
)

config = {"configurable": {"thread_id": "booking-001"}}

# 第一次调用：Agent 会请求调用工具，但执行前暂停
result = agent.invoke({"messages": [HumanMessage(content="帮我预订 Python3 基础教程，我叫小明")]}, config)
print("Agent 等待确认中...")
# 查看暂停时的状态
print(f"当前状态: {result.get('messages', [])[-1].tool_calls if hasattr(result.get('messages', [])[-1], 'tool_calls') else '无工具调用'}")

print("带暂停点的 Agent 已创建")

Agent 等待确认中...
当前状态: [{'name': 'book_course', 'args': {'course_name': 'Python3 基础教程', 'user_name': '小明'}, 'id': 'call_00_pe007vN49IiIws4uqNVW3867', 'type': 'tool_call'}]
带暂停点的 Agent 已创建


## 恢复执行

人工确认后，通过 `invoke(None, config)` 恢复 Agent 执行。

In [3]:
# 人工确认后恢复执行
result = agent.invoke(None, config)
print(f"最终回复: {result['messages'][-1].content}")

print("确认后通过 invoke(None, config) 恢复")

最终回复: ✅ 已经成功为你（**小明**）预订了 **《Python3 基础教程》**！请按时参加课程哦～如果还有其他需要，随时找我！😊
确认后通过 invoke(None, config) 恢复


**术语：**

- **interrupt_before**：在指定节点前暂停
- **interrupt_after**：在指定节点后暂停
- **Human-in-the-Loop**：人工介入机制
- **invoke(None, config)**：恢复暂停的 Agent 执行